# 05 · H3: ECG-specific vs generic pretraining augmentation

Protocol: `docs/h3_protocol_v1.md` (frozen, D37). Runs on **Kaggle** as a saved version (Save Version → Save & Run All).

### Inputs to attach (Add Input)
1. The private cache dataset (same as notebooks 03 and 04).
2. *Your Work* → the **output of notebook 04** (its `work.tar` holds the core grid with checkpoints, the SSL encoders
   and the corruption results). The output of notebook 03 also works; the core runs are then re-evaluated on the
   corruption suite here. Attach only one of the two.

Settings: **Accelerator GPU T4 x2**, **Internet on**. Expected time: about 3 hours.

### What it does
A. pretrains the seed-0 encoder of each policy-level (5 new; ecg-mid exists) ·
B. linear probes on fold 9 and selection of one level per policy ·
C. pretrains seeds 1–4 of the selected levels ·
D. fine-tunes in the C1 configuration at 100% ·
E. corruption suite on every H3 run.

Resumable: if a version times out, attach its output as input to the next version and run again.
Download `results_export_05.zip` from the Output tab afterwards.

In [ ]:
ALLOW_INTERACTIVE = False     # True only for short tests you are willing to lose (lab notebook, Failure 7)

In [ ]:
import os, glob, json, subprocess, sys, tarfile, zipfile
if not os.path.isdir('/kaggle/working'):
    raise RuntimeError('This notebook is written for Kaggle.')
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != 'Batch' and not ALLOW_INTERACTIVE:
    raise RuntimeError('Interactive session: results would be lost. Use Save Version -> Save & Run All (Commit).')
REPO, CACHE, WORK = '/tmp/robust-ecg-ssl', '/tmp/cache', '/tmp/work'
ARCHIVE, EXPORT = '/kaggle/working/work.tar', '/kaggle/working/results_export_05.zip'
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/ptbxl_X.npy', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'sph_X.npy'))]
if not hits:
    raise FileNotFoundError('No input dataset with ptbxl_X.npy and sph_X.npy: add the cache dataset.')
prior = sorted(glob.glob('/kaggle/input/**/work.tar', recursive=True), key=os.path.getsize)
if not prior:
    raise FileNotFoundError('No work.tar among the inputs: add the output of notebook 04 (or 03).')
if len(prior) > 1:
    print('several work.tar inputs found; using the largest:', prior[-1])
if not os.path.exists(WORK):
    with tarfile.open(prior[-1]) as t:
        t.extractall('/tmp')
    print('restored', prior[-1])
n_ckpt = len(glob.glob(f'{WORK}/runs/*/ckpt_best.pt')); n_enc = len(glob.glob(f'{WORK}/ssl/SSL_ecg-mid_s*/encoder.pt'))
print(n_ckpt, 'core checkpoints,', n_enc, 'core SSL encoders,', len(glob.glob(f'{WORK}/runs_h3/*/done.json')), 'H3 runs already done')
assert n_ckpt == 120 and n_enc == 5, 'expected the complete core grid (120 checkpoints, 5 encoders)'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Salako07/robust-ecg-ssl.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, f'{REPO}/src')
from robust_ecg import corrupt, h3
assert corrupt.FROZEN, 'corruption protocol not frozen'
assert (h3.PROBE_STEPS, h3.PROBE_LR) == (1000, 1e-2), 'repository is not at the frozen H3 protocol (D37): push first'
from robust_ecg.colab_utils import copy_dir_with_retry
copy_dir_with_retry(hits[0], CACHE, remount=None)
import torch
GPUS = max(1, torch.cuda.device_count())
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Run stages A–E
One line per finished job; per-job console logs are in `work/logs/`. The work folder is archived after every stage.

In [ ]:
cmd = [sys.executable, '-u', f'{REPO}/scripts/run_h3.py', '--cache', CACHE, '--work', WORK, '--gpus', str(GPUS),
       '--archive', ARCHIVE]
subprocess.run(cmd, check=True)

## Summary (descriptive)
Probe scores, the selection, and seed-level means. The inference is `scripts/bootstrap_h3.py`, run afterwards.

In [ ]:
import pandas as pd
print(pd.read_csv(f'{WORK}/h3/probe.csv').round(4).to_string(index=False))
print(pd.read_csv(f'{WORK}/h3/view_distortion.csv').round(4).to_string(index=False))
man = json.load(open(f'{WORK}/h3/manifest.json'))
rows = []
for r in man['runs']:
    d = json.load(open(f"{WORK}/{r['run_dir']}/done.json")); c = json.load(open(f"{WORK}/{r['run_dir']}/corruptions.json"))
    ho = [m['degradation'] for m in c['conditions'].values() if m['held_out']]
    rows.append(dict(policy=r['policy'], level=r['level'], seed=r['seed'], primary=r['in_primary'],
                     fold10=d['test_macro_auroc'], sph_e3=d['sph_e3_macro_auroc'], e3_degradation=d['e3_degradation'],
                     held_out_degradation=sum(ho) / len(ho)))
t = pd.DataFrame(rows)
print(t[t.primary].groupby(['policy', 'level']).mean(numeric_only=True).drop(columns=['seed', 'primary']).round(4).to_string())
print(t[t.seed == 0].drop(columns=['seed']).round(4).to_string(index=False))

## Export
Small files only (no checkpoints, no encoders). The full state is in `work.tar`.

In [ ]:
keep = ('config.json', 'log.csv', 'done.json', 'predictions.npz', 'corruptions.json', 'corruptions.npz')
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{WORK}/runs_h3/*/*') + glob.glob(f'{WORK}/ssl/*/*'):
        if f.endswith(keep):
            z.write(f, os.path.relpath(f, WORK))
    for r in man['runs']:                      # reused core runs: only the corruption files may be new
        if r['reused_core_run']:
            for name in ('corruptions.json', 'corruptions.npz'):
                z.write(f"{WORK}/{r['run_dir']}/{name}", f"{r['run_dir']}/{name}")
    for f in glob.glob(f'{WORK}/h3/*') + glob.glob(f'{WORK}/runs_h3/*.csv') + [f'{WORK}/ssl/registry_ssl.csv'] + \
             [f for f in glob.glob(f'{WORK}/logs/*.log') if 'h3-' in f or 'probe_' in f or 'SSL_' in f]:
        z.write(f, os.path.relpath(f, WORK))
print(EXPORT, round(os.path.getsize(EXPORT) / 1e6, 2), 'MB,', len(zipfile.ZipFile(EXPORT).namelist()), 'files')